# Italy-Ghana PINN study - local execution notebook
This notebook orchestrates the current repository scripts; scientific implementations remain in `src/`. It is designed for a local Windows/Jupyter or VS Code environment with a multi-core CPU, optional CUDA GPU, and sufficient RAM. A short benchmark helps choose the device. Training jobs run one at a time: these networks are small, and concurrent CUDA fits usually contend rather than improve throughput.


The primary comparison pairs Ghana and Italy-primary at origins 45, 60, and 75 with a 15-day horizon. Italy-secondary is a separate robustness arm. Main grids use the identity-bound, per-fit `checkpoint_grid_runner.py`, so completed work resumes without mixing different masters or settings. Keep each output path tied to the same data, device, origins, seeds, epochs, and collocation count.

The notebook reflects the current `src` methodology: source-date coverage is audited before Italy zero-filling; uncertain correction is not converted to zero; sparsity masks are nested and audited; synthetic Ghana hospital admissions are daily admission inflows, not occupancy or observations; the H ablation has a dedicated runner; and Rt extraction exports fitted PINN `R_net`. Repeated seeds are not treated as independent forecast origins, and cross-country comparisons are not causal claims.

In [ ]:
!nvidia-smi

## 1. Local runtime and resource policy

Run this notebook with the project environment selected as its kernel. The default policy selects CUDA when available; set `PREFERRED_DEVICE` to `"cpu"` or `"cuda"` after checking the benchmark. CPU fits use up to eight intra-op threads (leaving at least one logical CPU free); CUDA fits use one CPU thread and one training process to avoid oversubscription and GPU contention.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import time

# Find the repository from the notebook folder or set the path explicitly.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate /"src"/"pinn.py").is_file()),
    None, 
)
if PROJECT_ROOT is None:
    raise RuntimeError("Set PROJECT_ROOT to the repository containing src/pinn.py")
PROJECT_ROOT = PROJECT_ROOT.resolve()
os.chdir(PROJECT_ROOT)


CPU_LOGICAL = os.cpu_count() or 1
LOCAL_CPU_THREADS = max(1, min(10, CPU_LOGICAL - 1))
PREFERRED_DEVICE = "cpu"  # choose "cpu" or "cuda" to pin all subsequent training,
#                          and "auto" to use CUDA if available, otherwise CPU

# Torch reads these at import. The child processes launched below inherit the
# corresponding per-device environment, so settings apply to actual model runs.
os.environ["OMP_NUM_THREADS"] = str(LOCAL_CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(LOCAL_CPU_THREADS)
import torch

if PREFERRED_DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("PREFERRED_DEVICE must be auto, cpu, or cuda")
if PREFERRED_DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but this kernel has no available CUDA device")
ACTIVE_DEVICE = (
    "cuda" if PREFERRED_DEVICE == "auto" and torch.cuda.is_available()
    else "cpu" if PREFERRED_DEVICE == "auto"
    else PREFERRED_DEVICE
)
JOB_CPU_THREADS = 1 if ACTIVE_DEVICE == "cuda" else LOCAL_CPU_THREADS
torch.set_num_threads(JOB_CPU_THREADS)

# Full paper-scale settings for reported experiments; do not use reduced epochs
# for thesis results. The resumable grid runner writes each completed fit.
ORIGINS_PRIMARY = [45, 60, 75]
HORIZON = 15
SEEDS = [1, 2, 3]
EPOCHS_JOINT = 5000
EPOCHS_SPLIT_DATA = 3000
EPOCHS_SPLIT_ODE = 1000
COLLOCATION = 6000
MAX_FITS_PER_CALL = None  # e.g. 12 to stop cleanly and resume by rerunning the same cell
DEVICE_TAG = ACTIVE_DEVICE

TABLES = PROJECT_ROOT / "results" / "tables"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw"

def run_script(script, *args, device=None):
    """Run a repository script, stream its output, and return the captured log."""
    project_root = PROJECT_ROOT.resolve()

    selected_device = device or ACTIVE_DEVICE
    if selected_device not in {"cpu", "cuda"}:
        raise ValueError(f"Unsupported local device: {selected_device}")
    if selected_device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA run requested but CUDA is not available")

    path = project_root / "src" / script
    if not path.is_file():
        raise FileNotFoundError(path)

    env = os.environ.copy()
    threads = 1 if selected_device == "cuda" else LOCAL_CPU_THREADS
    env["OMP_NUM_THREADS"] = str(threads)
    env["MKL_NUM_THREADS"] = str(threads)
    if selected_device == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = ""
    else:
        env.pop("CUDA_VISIBLE_DEVICES", None)

    command = [sys.executable, "-u", str(path), *map(str, args)]
    print("Running:", subprocess.list2cmdline(command), flush=True)

    output_lines = []
    with subprocess.Popen(
        command,
        cwd=project_root,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
                output_lines.append(line)
        return_code = process.wait()

    if return_code:
        raise subprocess.CalledProcessError(
            return_code, command, output="".join(output_lines)
        )
    return "".join(output_lines)


def load_result_tables(out):
    """Load CSV or Parquet results from the output path or its directory."""

    target = Path(out)
    if target.is_file():
        files = [target]
    elif target.is_dir():
        files = sorted(
            p for p in target.rglob("*")
            if p.suffix.lower() in {".csv", ".parquet"}
        )
    else:
        files = sorted(
            p for p in target.parent.glob(target.name + "*")
            if p.suffix.lower() in {".csv", ".parquet"}
        )

    tables = {}
    for file in files:
        if file.suffix.lower() == ".csv":
            tables[file.stem] = pd.read_csv(file)
        else:
            tables[file.stem] = pd.read_parquet(file)

    if not tables:
        raise FileNotFoundError(
            f"No CSV or Parquet result files found for output path: {target}"
        )
    return tables

    # Merge errors into standard output and print each line as the script produces it.
    # Intended output: the command above, followed by the script's live log/progress.
    with subprocess.Popen(
        command,
        cwd=project_root,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()

    # Report script failures as a Python exception after displaying their output.
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def run_checkpoint_grid(country, master, regimes, mode, out, origins=None,
                        max_fits=MAX_FITS_PER_CALL):
    """Run/resume one audited country-mode grid on the selected local device."""
    fit_origins = origins or ORIGINS_PRIMARY
    args = [
        "--country", country,
        "--master", master,
        "--out", out,
        "--origins", *fit_origins,
        "--horizon", HORIZON,
        "--regimes", *regimes,
        "--modes", mode,
        "--seeds", *SEEDS,
        "--epochs-joint", EPOCHS_JOINT,
        "--epochs-split-data", EPOCHS_SPLIT_DATA,
        "--epochs-split-ode", EPOCHS_SPLIT_ODE,
        "--collocation", COLLOCATION,
    ]
    if max_fits is not None:
        if max_fits < 1:
            raise ValueError("MAX_FITS_PER_CALL must be positive or None")
        args.extend(["--max-fits", max_fits])
    run_script("checkpoint_grid_runner.py", *args, device=ACTIVE_DEVICE)

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Logical CPUs: {CPU_LOGICAL}; CPU threads/job: {JOB_CPU_THREADS}")
print(f"Selected training device: {ACTIVE_DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")



Project: D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-
Python: c:\Users\BrothermanKNAN\anaconda3\python.exe
Platform: Windows-11-10.0.29683-SP0
Logical CPUs: 8; CPU threads/job: 7
Selected training device: cpu
PyTorch: 2.12.1+cu126
CUDA available: True


In [2]:

if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_id)
        print(f"GPU {gpu_id}: {props.name}; VRAM={props.total_memory / 1024**3:.1f} GiB")
try:
    import psutil
    import pandas as pd
    memory = psutil.virtual_memory()
    print(f"RAM: {memory.total / 1024**3:.1f} GiB total; "
          f"{memory.available / 1024**3:.1f} GiB available")
except ImportError:
    print("RAM availability not reported (optional psutil package is not installed).")
print("Execution strategy: one training process at a time; no multi-process GPU oversubscription.")

GPU 0: NVIDIA GeForce MX250; VRAM=4.0 GiB


RAM: 15.8 GiB total; 2.9 GiB available
Execution strategy: one training process at a time; no multi-process GPU oversubscription.


## 2. Device benchmark (measurement only)

This short joint-fit benchmark compares one and several CPU thread counts with CUDA when available. It uses the Ghana master only to measure this machine; its small epoch count is **not** a scientific run. Let the first run initialize each device, measure repeated trials, then use the median. Tiny networks may not benefit from many CPU threads; prefer measured wall-clock time over core-count assumptions.

In [2]:
import numpy as np
import pandas as pd
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pinn_old import Config, ReducedSIRPINN

benchmark_master_path = PROCESSED / "ghana_national_master.csv"
if not benchmark_master_path.is_file():
    raise FileNotFoundError(
        f"Prepare Ghana first so the benchmark can read {benchmark_master_path}"
    )
benchmark_master = pd.read_csv(benchmark_master_path, index_col=0, parse_dates=True)
benchmark_I = benchmark_master["I_proxy"].to_numpy(dtype=float)
benchmark_origin = min(75, len(benchmark_I) - HORIZON)
if benchmark_origin < 20:
    raise ValueError("Ghana master is too short for the local device benchmark")
benchmark_scale = float(benchmark_I[:benchmark_origin].max())
if benchmark_scale <= 0:
    raise ValueError("Benchmark training prefix must contain a positive I_proxy value")
t_data = torch.tensor(
    np.arange(benchmark_origin) / benchmark_origin, dtype=torch.float32
).reshape(-1, 1)
i_obs = torch.tensor(
    benchmark_I[:benchmark_origin] / benchmark_scale, dtype=torch.float32
).reshape(-1, 1)

candidates = [("cpu", count) for count in sorted({1, min(4, LOCAL_CPU_THREADS), LOCAL_CPU_THREADS})]
if torch.cuda.is_available():
    candidates.append(("cuda", 1))
benchmark_rows = []
for device, thread_count in candidates:
    torch.set_num_threads(thread_count)
    # Untimed warm-up avoids charging CUDA/context initialization to the trials.
    warmup = ReducedSIRPINN(
        Config(epochs_joint=3, n_collocation=500, device=device, seed=34),
        tf=benchmark_origin, scale_I=benchmark_scale,
    )
    warmup.fit_joint(t_data, i_obs)
    if device == "cuda":
        torch.cuda.synchronize()
    trial_seconds = []
    for repeat in range(2):
        model = ReducedSIRPINN(
            Config(epochs_joint=60, n_collocation=6000, device=device, seed=34),
            tf=benchmark_origin, scale_I=benchmark_scale,
        )
        start = time.perf_counter()
        model.fit_joint(t_data, i_obs)
        if device == "cuda":
            torch.cuda.synchronize()
        trial_seconds.append(time.perf_counter() - start)
    benchmark_rows.append({
        "device": device,
        "cpu_threads": thread_count,
        "seconds_per_60_epochs_median": float(np.median(trial_seconds)),
        "estimated_full_joint_fit_minutes": (
            float(np.median(trial_seconds)) * EPOCHS_JOINT / 60 / 60
        ),
    })

torch.set_num_threads(JOB_CPU_THREADS)
benchmark_results = pd.DataFrame(benchmark_rows).sort_values(
    "seconds_per_60_epochs_median"
)
benchmark_results

,device,cpu_threads,seconds_per_60_epochs_median,estimated_full_joint_fit_minutes
0,cpu,1,61.001030,84.723653
1,cpu,4,76.726404,106.564450
3,cuda,1,80.887651,112.343959
2,cpu,7,104.058010,144.525014


## 3. PINN implementation validation gate

Validate the shared infection-only `ReducedSIRPINN` against the repository's Millevoi Case 4 reference before country grids. Split and joint validations are separate so each result is explicit in the JSON sidecar. These are full paper-scale fits; run them only after selecting the device based on the benchmark. A successful Case 4 check is necessary, not proof that country-specific assumptions are valid.

In [ ]:
# # Fast smoke test: validates that loss, gradients, outputs and JSON saving work.
# run_script('pinn.py', 
#   "--repo", "reference",
#   "--out", "results/tables/dev_case4.json",
#   "--mode", "both",
#   "--epochs-joint", 500,
#   "--epochs-split-data", 500,
#   "--epochs-split-ode", 300,
#   "--collocation", 1700,
#   "--seed", 34
#   )

Running: c:\Users\BrothermanKNAN\anaconda3\python.exe -u "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\src\pinn.py" --repo reference --out results/tables/dev_case4.json --mode both --epochs-joint 500 --epochs-split-data 500 --epochs-split-ode 300 --collocation 1700 --seed 34
[joint] err_I=0.9999 (paper 0.1411)  err_Rt=1.9695 (paper 0.4961)  308.5s
[split] err_I=0.1221 (paper 0.1331)  err_Rt=0.4709 (paper 0.4744)  136.0s


'[joint] err_I=0.9999 (paper 0.1411)  err_Rt=1.9695 (paper 0.4961)  308.5s\n[split] err_I=0.1221 (paper 0.1331)  err_Rt=0.4709 (paper 0.4744)  136.0s\n'

In [ ]:
run_script(
    "pinn.py",
    "--repo", "reference",
    "--out", TABLES / "validation_case4_local.json",
    "--mode", "both",
    "--epochs-joint", EPOCHS_JOINT,
    "--epochs-split-data", EPOCHS_SPLIT_DATA,
    "--epochs-split-ode", EPOCHS_SPLIT_ODE,
    "--collocation", COLLOCATION,
    "--seed", 34,
)

Running: c:\Users\BrothermanKNAN\anaconda3\python.exe -u "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\src\pinn.py" --repo reference --out "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\results\tables\validation_case4_local.json" --mode both --epochs-split-data 300 --epochs-split-ode 100 --collocation 500 --seed 1
[joint] err_I=0.5368 (paper 0.1411)  err_Rt=0.5060 (paper 0.4961)  787.2s
[split] err_I=0.2277 (paper 0.1331)  err_Rt=0.5488 (paper 0.4744)  24.7s


'[joint] err_I=0.5368 (paper 0.1411)  err_Rt=0.5060 (paper 0.4961)  787.2s\n[split] err_I=0.2277 (paper 0.1331)  err_Rt=0.5488 (paper 0.4744)  24.7s\n'

In [ ]:
# #split mode test: cheaper than joint mode, but still validates that loss, gradients, outputs and JSON saving work.
Uncomment the following to run a split-mode test. 
It is cheaper than joint mode, but still validates that loss, gradients, outputs and JSON saving work.
# run_script(
#     "pinn.py",
#     "--repo", "reference",
#     "--out", TABLES / "validation_case4_local.json",
#     "--mode", "split",
#     "--epochs-split-data", EPOCHS_SPLIT_DATA,
#     "--epochs-split-ode", EPOCHS_SPLIT_ODE,
#     "--collocation", COLLOCATION,
#     "--seed", SEED,
# )

Running: c:\Users\BrothermanKNAN\anaconda3\python.exe -u "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\src\pinn.py" --repo reference --out "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\results\tables\validation_case4_local.json" --mode split --epochs-split-data 300 --epochs-split-ode 100 --collocation 500 --seed 1
[split] err_I=0.2277 (paper 0.1331)  err_Rt=0.5488 (paper 0.4744)  41.9s


'[split] err_I=0.2277 (paper 0.1331)  err_Rt=0.5488 (paper 0.4744)  41.9s\n'

## 4. Preprocessing and synthesis

By default, `REFRESH_MASTERS = True` rebuilds the processed masters from the current raw inputs. Ghana outputs are first copied to a timestamped backup folder; Italy preparation uses explicit `--overwrite` and retains its own backups. Set the flag to `False` only when the existing masters, audit sidecars, and sparsity masks have been confirmed current. The Italy-secondary master is prepared with `--ascertainment none` so the post-cutoff input remains the raw observed incidence and the data-quality figure can validate it. Sparsification runs after both masters are produced. The Ghana synthetic-admissions file is regenerated from the current Ghana master with an audit sidecar.

In [ ]:
from datetime import datetime
import json
import shutil
import numpy as np
import pandas as pd

RAW_GHANA = RAW / "ghana_hera.csv"
RAW_ITALY = RAW / "italy_case_hosp.csv"
REFRESH_ITALY_SOURCE = False
REFRESH_MASTERS = True  # rebuild from the current raw inputs, with backups

if REFRESH_ITALY_SOURCE and RAW_ITALY.is_file():
    raw_backup = PROJECT_ROOT / "results" / "preprocess_backups" / datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_backup.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RAW_ITALY, raw_backup / RAW_ITALY.name)
if REFRESH_ITALY_SOURCE or not RAW_ITALY.is_file():
    run_script(
        "build_italy_national.py",
        "--mode", "bydate",
        "--confirmed", RAW / "iss_bydate_italy_confirmed.csv",
        "--hospitalisations", RAW / "iss_bydate_italy_hospitalizations.csv",
        "--out", RAW_ITALY,
    )
if not RAW_GHANA.is_file() or not RAW_ITALY.is_file():
    raise FileNotFoundError("Required raw Ghana/Italy source CSVs are missing under data/raw")

if REFRESH_MASTERS:
    backup_dir = PROJECT_ROOT / "results" / "preprocess_backups" / datetime.now().strftime("%Y%m%d_%H%M%S")
    ghana_outputs = [
        "ghana_national_master.csv", "ghana_national_excl_unspecified.csv",
        "ghana_weekly.csv", "revision_log.csv", "ghana_millevoi_format.csv",
        "audit_summary.json",
    ]
    for name in ghana_outputs:
        old_file = PROCESSED / name
        if old_file.is_file():
            backup_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_file, backup_dir / name)
    run_script("prepare_ghana.py", "--input", RAW_GHANA, "--outdir", PROCESSED)
    
    # Primary is the fixed r=6 benchmark. Secondary uses raw reporting throughout;
    # do not turn unknown post-cutoff correction into zeros.
    run_script(
        "prepare_italy.py", "--arm", "primary", "--input", RAW_ITALY,
        "--outdir", PROCESSED, "--overwrite",
    )
    run_script(
        "prepare_italy.py", "--arm", "secondary", "--ascertainment", "none",
        "--input", RAW_ITALY, "--outdir", PROCESSED, "--overwrite",
    )
    for country in ("italy_primary", "italy_secondary"):
        run_script(
            "sparsify.py", "--master", PROCESSED / f"{country}_national_master.csv",
            "--country", country, "--pct", 20, 30, 40, "--inplace", "--force",
        )

# Always regenerate derived H from the current Ghana master using the updated equations.
ghana_master_path = PROCESSED / "ghana_national_master.csv"
hospitalisation_path = PROCESSED / "ghana_hospitalisation_synth.csv"
run_script(
    "synth_hospitalisation.py", "--master", ghana_master_path,
    "--out", hospitalisation_path,
)

# Audit the saved products themselves before any grid is allowed to start.
required_files = [
    ghana_master_path,
    PROCESSED / "italy_primary_national_master.csv",
    PROCESSED / "italy_secondary_national_master.csv",
    hospitalisation_path,
    hospitalisation_path.with_suffix(".meta.json"),
]
missing = [file for file in required_files if not file.is_file()]
if missing:
    raise FileNotFoundError(f"Missing processed products: {missing}. Set REFRESH_MASTERS=True.")
secondary_audit = json.loads(
    (PROCESSED / "italy_secondary_audit_summary.json").read_text(encoding="utf-8")
)
if secondary_audit.get("ascertainment_mode") != "none":
    raise ValueError("Italy-secondary must be regenerated with --ascertainment none")
primary_audit = json.loads(
    (PROCESSED / "italy_primary_audit_summary.json").read_text(encoding="utf-8")
)
if primary_audit.get("ascertainment_mode") != "fixed":
    raise ValueError("Italy-primary must retain the fixed r=6 primary correction")
for country in ("italy_primary", "italy_secondary"):
    master = pd.read_csv(PROCESSED / f"{country}_national_master.csv", index_col=0, parse_dates=True)
    if master.empty or "country" not in master or set(master["country"].dropna()) != {country}:
        raise ValueError(f"{country} master identity/coverage audit failed")
    mask_arrays = {}
    for pct in (20, 30, 40):
        column = f"observed_sparse{pct}"
        if column not in master or master[column].isna().any() or not master[column].isin([0, 1]).all():
            raise ValueError(f"{country} {column} is missing or non-binary")
        values = master[column].to_numpy(dtype=np.int8)
        if int((values == 0).sum()) != round(len(master) * pct / 100):
            raise ValueError(f"{country} {column} has an unexpected full-arm dropout count")
        mask_arrays[pct] = values
    if (mask_arrays[30] > mask_arrays[20]).any() or (mask_arrays[40] > mask_arrays[30]).any():
        raise ValueError(f"{country} sparsity masks are not nested")
synthesis_audit = json.loads(
    hospitalisation_path.with_suffix(".meta.json").read_text(encoding="utf-8")
)
if synthesis_audit.get("schema_version") != 2 or synthesis_audit.get("audit_status") != "passed":
    raise ValueError("Synthetic H artifact is stale or its current audit failed")
print("Preprocessing, country identity, mask, and synthesis audits passed.")

Running: c:\Users\BrothermanKNAN\anaconda3\python.exe -u "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\src\prepare_ghana.py" --input "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\data\raw\ghana_hera.csv" --outdir "D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-\data\processed"
{
  "date_start": "2020-03-12",
  "date_end": "2021-05-19",
  "n_days": 434,
  "n_regions": 17,
  "total_cases": 93674,
  "total_deaths": 783,
  "total_recoveries": 91647,
  "observed_days_simple": 317,
  "observed_pct_simple": 73.0,
  "observed_days_context": 330,
  "unreported_days": 117,
  "longest_gap": 14,
  "n_gaps": 70,
  "negative_records_clipped": 8,
  "cases_excl_unspecified": 91946,
  "primary_infectious_period_D": 5,
  "n_weeks": 62
}

Wrote:
  audit_summary.json
  ghana_hospitalisation_synth.csv
  ghana_hospitalisation_synth.meta.json
  ghana_millevoi_format.csv
  ghana_national_excl_unspecified.csv
  ghana_national_master.csv

## 5. Primary forecast grids - resumable and device-specific

Run split first, then joint after reviewing split outputs. Each cell runs one sequential checkpoint runner, which atomically saves each completed fit and writes an experiment-identity sidecar. If interrupted, rerun the same cell and command. Device-specific filenames keep CPU and GPU results separate. Leave `MAX_FITS_PER_CALL = None` for a complete local run, or set a positive limit to checkpoint manageable chunks. The common Italy-primary origins and horizon are validated by the runner; do not increase origins beyond the primary window.

In [ ]:
ghana_split_path = TABLES / f"grid_ghana_split_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "ghana", PROCESSED / "ghana_national_master.csv",
    ["daily", "weekly", "mask"], "split", ghana_split_path,
)

In [ ]:
italy_primary_split_path = TABLES / f"grid_italy_primary_split_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "italy_primary", PROCESSED / "italy_primary_national_master.csv",
    ["daily", "weekly", "sparse20", "sparse30", "sparse40"],
    "split", italy_primary_split_path,
)

In [ ]:
ghana_joint_path = TABLES / f"grid_ghana_joint_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "ghana", PROCESSED / "ghana_national_master.csv",
    ["daily", "weekly", "mask"], "joint", ghana_joint_path,
)

In [ ]:
italy_primary_joint_path = TABLES / f"grid_italy_primary_joint_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "italy_primary", PROCESSED / "italy_primary_national_master.csv",
    ["daily", "weekly", "sparse20", "sparse30", "sparse40"],
    "joint", italy_primary_joint_path,
)

### Optional: Italy-secondary robustness arm

This is not part of the primary Ghana-Italy comparison. It uses a separate country label, master, output path, and origins spanning the longer series. Keep it separate in metrics and reporting. The 15-day forecast windows below fit the current 999-day secondary master; recheck after changing source dates.

In [ ]:
SECONDARY_ORIGINS = [100, 300, 500, 700, 900]
secondary_master_path = PROCESSED / "italy_secondary_national_master.csv"
secondary_master = pd.read_csv(secondary_master_path, index_col=0, parse_dates=True)
if any(origin + HORIZON > len(secondary_master) for origin in SECONDARY_ORIGINS):
    raise ValueError("One or more predeclared Italy-secondary origins do not fit the current master")
italy_secondary_split_path = TABLES / f"grid_italy_secondary_split_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "italy_secondary", secondary_master_path,
    ["daily", "weekly", "sparse20", "sparse30", "sparse40"],
    "split", italy_secondary_split_path, origins=SECONDARY_ORIGINS,
)

In [ ]:
italy_secondary_joint_path = TABLES / f"grid_italy_secondary_joint_{DEVICE_TAG}.csv"
run_checkpoint_grid(
    "italy_secondary", secondary_master_path,
    ["daily", "weekly", "sparse20", "sparse30", "sparse40"],
    "joint", italy_secondary_joint_path, origins=SECONDARY_ORIGINS,
)

## 6. Ghana hospitalisation ablation (optional, separate experiment)

The dedicated runner compares matched infection-only and auxiliary-H models. The H target is the synthetic **daily admissions** channel generated in Stage 4, with a documented 14-day incidence-based infectious-prevalence approximation and Dwomoh Q equation. It is not observed hospital data; H occupancy is diagnostic only and is not fitted. The runner checks hashes, channel validity, origin/regime/mode/seed pairing, and training-only normalization. This fit set is compute-intensive and is not resumable per fit, so keep it separate from the checkpointed primary grids.

In [ ]:
H_ABLATION_MODES = ["split"]
H_ABLATION_SCENARIOS = ["dwomoh_s2_s3", "dwomoh_s4_s5"]
hospitalisation_ablation_path = TABLES / f"ghana_hospitalisation_ablation_{DEVICE_TAG}.csv"
run_script(
    "run_ghana_hospitalisation_ablation.py",
    "--master", PROCESSED / "ghana_national_master.csv",
    "--hospitalisation", hospitalisation_path,
    "--out", hospitalisation_ablation_path,
    "--origins", *ORIGINS_PRIMARY,
    "--horizon", HORIZON,
    "--regimes", "daily", "weekly", "mask",
    "--modes", *H_ABLATION_MODES,
    "--seeds", *SEEDS,
    "--scenarios", *H_ABLATION_SCENARIOS,
    "--epochs-joint", EPOCHS_JOINT,
    "--epochs-split-data", EPOCHS_SPLIT_DATA,
    "--epochs-split-ode", EPOCHS_SPLIT_ODE,
    "--collocation", COLLOCATION,
)

## 7. Combine completed primary grids and run paired summaries

Do not merge until all four primary split/joint grids are complete. The merge below refuses missing files, wrong country labels, or duplicate keys. Only Ghana and Italy-primary enter the core comparison; secondary remains a separate robustness result. `equity_metrics.py` pairs on common origins and emits absolute/log-ratio MASE degradation with denominator safeguards. Statistical inference from only three origins is explicitly exploratory; repeated seeds do not increase the independent-origin count.

In [ ]:
import pandas as pd


def load_grid_parts(paths, expected_country):
    frames = []
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f"Grid is missing or incomplete: {path}")
        frame = pd.read_csv(path)
        if frame.empty or "country" not in frame:
            raise ValueError(f"Grid has no rows or country label: {path}")
        if set(frame["country"].astype(str)) != {expected_country}:
            raise ValueError(f"Unexpected country labels in {path}: {set(frame['country'])}")
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    keys = ["country", "experiment_id", "origin", "regime", "method", "seed"]
    if combined.duplicated(keys).any():
        raise ValueError(f"Duplicate result keys found for {expected_country}")
    return combined


ghana_grid = load_grid_parts([ghana_split_path, ghana_joint_path], "ghana")
italy_primary_grid = load_grid_parts(
    [italy_primary_split_path, italy_primary_joint_path], "italy_primary"
)
TABLES.mkdir(parents=True, exist_ok=True)
ghana_grid_path = TABLES / "grid_ghana_primary.csv"
italy_primary_grid_path = TABLES / "grid_italy_primary.csv"
ghana_grid.to_csv(ghana_grid_path, index=False)
italy_primary_grid.to_csv(italy_primary_grid_path, index=False)
print("Ghana rows:", len(ghana_grid), "Italy-primary rows:", len(italy_primary_grid))

In [ ]:
equity_path = TABLES / "equity_summary_primary.csv"
run_script(
    "equity_metrics.py",
    "--grid", ghana_grid_path,
    "--italy-grid", italy_primary_grid_path,
    "--out", equity_path,
)
run_script(
    "stat_tests.py",
    "--grid", ghana_grid_path,
    "--italy-grid", italy_primary_grid_path,
    "--out", TABLES / "significance_primary.csv",
    "--italy-arm", "primary",
)

## 8. Learned Rt trajectories (optional refits)

`extract_rt.py` refits the reduced-SIR PINN on each complete country series and exports its learned `R_net(t)`. This is not the old finite-difference proxy. The refits are separate from the expanding-origin forecasting grids; their outputs must be described as full-series fitted curves. Select a modest seed set and use the same `ACTIVE_DEVICE` policy. This does not make the exploratory estimates causal or equivalent to independent surveillance Rt estimates.

In [ ]:
for country, master_path in [
    ("ghana", PROCESSED / "ghana_national_master.csv"),
    ("italy_primary", PROCESSED / "italy_primary_national_master.csv"),
]:
    run_script(
        "extract_rt.py",
        "--master", master_path,
        "--country", country,
        "--outdir", PROJECT_ROOT / "results" / "rt",
        "--modes", "split",
        "--seeds", *SEEDS,
        "--epochs-joint", EPOCHS_JOINT,
        "--epochs-split-data", EPOCHS_SPLIT_DATA,
        "--epochs-split-ode", EPOCHS_SPLIT_ODE,
        "--collocation", COLLOCATION,
        "--device", ACTIVE_DEVICE,
    )

## 9. Data-quality figure and metric visualization

The data-quality script validates each available master and specifically requires a current, uncorrupted Italy-secondary proxy. The secondary panel is intentionally distinct from Italy-primary. The comparison plot uses explicit `ghana` and `italy_primary` labels and absolute MASE degradation; it does not reinterpret missing percentages as causal effects.

In [ ]:
run_script("fig_dataquality.py")

In [ ]:
import matplotlib.pyplot as plt

combined_primary = pd.concat([ghana_grid, italy_primary_grid], ignore_index=True)
pinn_rows = combined_primary[
    combined_primary["method"].astype(str).str.startswith("pinn_")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
regime_order = ["daily", "weekly", "mask", "sparse20", "sparse30", "sparse40"]
for country, label, marker in [
    ("ghana", "Ghana", "o"),
    ("italy_primary", "Italy-primary", "s"),
]:
    part = pinn_rows[(pinn_rows["country"] == country) & (pinn_rows["method"] == "pinn_split")]
    present = [regime for regime in regime_order if regime in set(part["regime"])]
    medians = [part.loc[part["regime"] == regime, "mase"].median() for regime in present]
    axes[0].plot(present, medians, marker=marker, label=label)
axes[0].set_ylabel("Median MASE")
axes[0].set_xlabel("Observation regime")
axes[0].set_title("Split-fit forecasting error by regime")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(frameon=False)

equity = pd.read_csv(equity_path)
if not {"country", "regime", "mase_gap"}.issubset(equity.columns):
    raise ValueError("Equity output lacks the expected absolute MASE-gap fields")
summary = equity.groupby(["country", "regime"])["mase_gap"].median().unstack("country")
summary.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("Median absolute MASE gap vs daily")
axes[1].set_xlabel("Observation regime")
axes[1].set_title("Paired-origin degradation (descriptive)")
axes[1].tick_params(axis="x", rotation=30)
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
figure_path = PROJECT_ROOT / "results" / "figures" / "fig_italy_ghana_equity_local.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180)
plt.show()
print("Saved:", figure_path)

## 10. Package outputs

The archive contains generated results only (tables, figures, Rt outputs, and audit sidecars), not source data or secrets. Review its contents before sharing.

In [ ]:
import shutil

results_dir = PROJECT_ROOT / "results"
if not results_dir.is_dir():
    raise FileNotFoundError(f"No results directory to archive: {results_dir}")
archive_base = PROJECT_ROOT / "thesis_results_local"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", results_dir))
print("Results archive:", archive_path)